In [ ]:
from pathlib import Path

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pymc as pm
import seaborn as sns

from climate_attitudes import configure_mpl
from climate_attitudes.dataset import Dataset
from climate_attitudes.settings import Config

FONT_PATH = Path("../fonts")
configure_mpl(FONT_PATH)

plt.rcParams["figure.constrained_layout.use"] = True

plt.rc("figure", dpi=150)

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(config)

In [ ]:
resp = (
    dataset.response.with_columns(
        pl.col("cc1").replace({99: 2}),
        (pl.col("dem_age") - pl.col("dem_age").mean()) / pl.col("dem_age").std(),
        pl.col(r"^Variant_cc(Solving|Compensation)$").replace({0: None}).log(),
    )
    .with_columns(
        (
            pl.col(r"^Variant_cc(Solving|Compensation)$")
            - pl.col(r"^Variant_cc(Solving|Compensation)$").mean()
        )
        / pl.col(r"^Variant_cc(Solving|Compensation)$").std()
    )
    .filter(pl.col("cc1").is_not_null())
)

In [ ]:
infer_data = resp.select(
    "participant_id",
    "wave",
    "dem_age",
    "dem_income",
    "cc1",
).collect()

age = infer_data.select("dem_age").to_numpy().flatten()
cc1 = infer_data.select("cc1").to_numpy().flatten()
income = infer_data.select("dem_income").to_numpy().flatten()

In [ ]:
# infer_data = (
#     dataset.response
#     .select(
#         "participant_id", "wave",
#         "dem_age",
#         "dem_income",
#         pl.col("cc1").replace({99: 2}),
#     )
#     .filter(pl.col("cc1").is_not_null())
#     .collect()
# )

# age = infer_data.select("dem_age").to_numpy().flatten()
# age = (age - age.mean()) / age.std()

# cc1 = infer_data.select("cc1").to_numpy().flatten()

In [ ]:
def fit_model(data: pl.DataFrame, column_name: str):
    N_RESPONSE_CLASSES = 5
    coords = {
        "cutpoints": np.arange(1, N_RESPONSE_CLASSES).astype(int),
        "observation": np.arange(len(data)),
        "cc1_response": ["No", "Yes", "Don't know"],
    }

    with pm.Model(coords=coords) as model:
        log_cost = data.select(f"Variant_{column_name}").to_numpy().flatten()
        # log_cost = (log_cost - log_cost.mean()) / log_cost.std()

        cc1 = pm.Data("cc1", data.select("cc1").to_numpy().flatten())

        age = data.select("dem_age").to_numpy().flatten()

        income = data.select("dem_income").to_numpy().flatten()

        cutpoints = pm.Normal(
            "alpha",
            mu=0,
            sigma=1,
            transform=pm.distributions.transforms.ordered,
            shape=N_RESPONSE_CLASSES - 1,
            initval=np.arange(N_RESPONSE_CLASSES - 1)
            - 2.5,  # use ordering (with coarse log-odds centering) for init
            dims="cutpoints",
        )

        beta_cost = pm.Normal("beta_cost", 0, 0.5)
        beta_age = pm.Normal("beta_age", 0, 0.5)
        beta_cc1 = pm.Normal("beta_cc1", 0, 0.5, dims="cc1_response")
        beta_income = pm.Normal("beta_income", 0, 0.5)

        eta = pm.Deterministic(
            "eta",
            beta_cost * log_cost
            + beta_age * age
            + beta_cc1[cc1]
            + beta_income * income,
        )

        _y = pm.OrderedLogistic(
            "y",
            eta=eta,
            cutpoints=cutpoints,
            observed=data.select(column_name).to_numpy().flatten() - 1,
        )

        trace = pm.sample_prior_predictive()
        trace.extend(pm.sample(2000, tune=1000))

    return model, trace

In [ ]:
def willingness_to_pay(trace, age=None, cc1=None, income=None):
    samples = az.extract(trace.posterior)
    if age is None or cc1 is None:
        age = trace.constant_data.age.values
        cc1 = trace.constant_data.cc1.values
        income = trace.constant_data.income.values
    neutral = (samples.alpha[1] + samples.alpha[2]) / 2
    log_wtp = (
        neutral.values
        - samples.beta_age.values * age[:, None]
        - samples.beta_cc1.values[cc1]
        - samples.beta_income.values * income[:, None]
    ) / samples.beta_cost.values
    return np.exp(log_wtp).mean(axis=1)

# ccSolve

In [ ]:
data = (
    resp.select(
        "participant_id",
        "wave",
        "dem_age",
        "dem_income",
        "cc1",
        "ccSolving",
        "Variant_ccSolving",
    )
    .filter(
        pl.col("ccSolving").is_not_null(),
        pl.col("Variant_ccSolving").is_not_null(),
    )
    .collect()
)

In [ ]:
m_cc_solve, trace_cc_solve = fit_model(data, "ccSolving")

In [ ]:
az.plot_trace(
    trace_cc_solve,
    var_names=["beta_cost", "beta_age", "beta_cc1", "beta_income", "alpha"],
    legend=True,
);

In [ ]:
wtp_cc_solve = willingness_to_pay(trace_cc_solve, age, cc1, income)
sns.displot(wtp_cc_solve)

# ccCompensation

In [ ]:
data = (
    resp.select(
        "participant_id",
        "wave",
        "dem_age",
        "dem_income",
        "cc1",
        "ccCompensation",
        "Variant_ccCompensation",
    )
    .filter(pl.col("ccCompensation").is_not_null())
    .filter(pl.col("Variant_ccCompensation").is_not_null())
    .collect()
)

In [ ]:
m_cc_comp, trace_cc_comp = fit_model(data, "ccCompensation")

In [ ]:
az.plot_trace(
    trace_cc_comp,
    var_names=["beta_cost", "beta_age", "beta_cc1", "beta_income", "alpha"],
    legend=True,
);

In [ ]:
wtp_cc_comp = willingness_to_pay(trace_cc_comp, age, cc1, income)
sns.displot(wtp_cc_comp)

In [ ]:
plot_data = infer_data.with_columns(wtp_cc_solve=wtp_cc_solve, wtp_cc_comp=wtp_cc_comp)

In [ ]:
sns.relplot(
    plot_data, x="wtp_cc_solve", y="wtp_cc_comp", s=5, hue="dem_age"
)  # , hue="", s=5)
# plt.xlim(0,1)
# plt.ylim(0,0.5)